# SGLD based light YOLO

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow_datasets as tfds

# データ読み込み

In [2]:
IMG_SIZE = 128
BATCH_SIZE = 4
NUM_CLASSES = 20

In [11]:
def preprocess(example):
    image = tf.image.resize(example['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0

    bbox = example['objects']['bbox'][0]
    label = example['objects']['label'][0]

    ymin, xmin, ymax, xmax = bbox
    center_x = (xmin + xmax) / 2
    center_y = (ymin + xmax) / 2
    width = xmax - xmin
    height = ymax - ymin
    yolo_label = tf.stack([center_x, center_y, width, height, 1.0])

    grid_size = 8
    y_true = tf.zeros((grid_size, grid_size, 5))
    grid_x = tf.cast(center_x * grid_size, tf.in32)
    grid_y = tf.cast(center_y * grid_size, tf.int32)
    y_true = tf.tensor_scatter_nd_update(y_true, [[grid_y, grid_x]], [yolo_label])
    return image, y_true

In [8]:
ds = tfds.load('voc/2012', split='train', shuffle_files=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating test examples...: 0 examples [00:00, ? examples/s]

...

Shuffling /home/fumish/tensorflow_datasets/voc/2012/incomplete.Y74LBS_5.0.0/voc-train.tfrecord*...:   0%|     …

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /home/fumish/tensorflow_datasets/voc/2012/incomplete.Y74LBS_5.0.0/voc-validation.tfrecord*...:   0%|…

Dataset voc downloaded and prepared to /home/fumish/tensorflow_datasets/voc/2012/5.0.0. Subsequent calls will reuse this data.


I0000 00:00:1760450207.892295 3021791 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7520 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:01:00.0, compute capability: 6.1


In [12]:
ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

OperatorNotAllowedInGraphError: in user code:

    File "/tmp/ipykernel_3021791/1437711562.py", line 8, in preprocess  *
        ymin, xmin, ymax, xmax = bbox

    OperatorNotAllowedInGraphError: Iterating over a symbolic `tf.Tensor` is not allowed. You can attempt the following resolutions to the problem: If you are running in Graph mode, use Eager execution mode or decorate this function with @tf.function. If you are using AutoGraph, you can try decorating this function with @tf.function. If that does not work, then you may be using an unsupported feature or your source code may not be visible to AutoGraph. See https://github.com/tensorflow/tensorflow/blob/master/tensorflow/python/autograph/g3doc/reference/limitations.md#access-to-source-code for more information.


# Model and loss

In [3]:
def conv_block(x, filters, kernel=3, stride=1):
    x = layers.Conv2D(filters, kernel, stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    return x

In [4]:
def tiny_yolo_like(input_shape=(128, 128, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    x = conv_block(inputs, 16)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 32)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 64)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 128)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 256)

    x = layers.Conv2D(num_classes * 5, 1, padding='same')(x)
    outputs = layers.Activation('sigmoid')(x)
    return Model(inputs, outputs)

In [5]:
model = tiny_yolo_like()

I0000 00:00:1760352697.160567 2969625 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7520 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:01:00.0, compute capability: 6.1


In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 128, 128, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 5)        │         1,285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 8, 8, 5)        │             0 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 395,877 (1.51 MB)

 Trainable params: 394,885 (1.51 MB)

 Non-trainable params: 992 (3.88 KB)

In [7]:
def yolo_loss(y_true, y_pred):
    coord_loss = tf.reduce_mean(tf.square(y_true[..., :4] - y_pred[..., :4]))
    obj_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(
        y_true[..., 4:], y_pred[..., 4:]
    ))
    return coord_loss + obj_loss

# SGLD

In [8]:
lr = 1e-5
num_steps = 2000
num_samples = num_steps // 10

In [9]:
trainable_vars = model.trainable_variables

In [10]:
samples_ta = tf.TensorArray(dtype=tf.float32, size=num_samples)
sample_idx = 0

In [11]:
@tf.function
def sgld_step(x, y):
    with tf.GradientTape() as tape:
        pred = model(x, training=True)
        loss = yolo_loss(y, pred)
    grads = tape.gradient(loss, trainable_vars)
    for var, grad in zip(trainable_vars, grads):
        noise = tf.random.normal(shape=var.shape, stddev=tf.sqrt(lr))
        var.assign_sub(0.5 * lr * grad - noise)
    return loss

In [12]:
for step in range(num_steps):
    x_batch = tf.random.normal([4, 128, 128, 3])
    y_batch = tf.random.uniform([4, 8, 8, 5], 0, 1)

    loss = sgld_step(x_batch, y_batch)
    if step % 10 == 0:
        print(f"step {step}, loss {loss.numpy():.4f}")
        flat = tf.concat([tf.reshape(v, [-1]) for v in trainable_vars], axis=0)
        samples_ta = samples_ta.write(sample_idx, flat)
        sample_idx += 1
samples = samples_ta.stack()

I0000 00:00:1760352708.883759 2972670 cuda_dnn.cc:529] Loaded cuDNN version 90600


step 0, loss 0.9170
step 10, loss 0.9907
step 20, loss 0.9391
step 30, loss 0.9350
step 40, loss 0.9985
step 50, loss 0.9627
step 60, loss 0.9520
step 70, loss 0.9929
step 80, loss 0.9866
step 90, loss 0.9567
step 100, loss 0.9582
step 110, loss 0.9471
step 120, loss 0.9498
step 130, loss 0.9475
step 140, loss 0.9402
step 150, loss 0.9379
step 160, loss 0.9245
step 170, loss 0.9281
step 180, loss 0.9375
step 190, loss 0.9659
step 200, loss 0.9171
step 210, loss 0.9368
step 220, loss 0.9724
step 230, loss 0.9551
step 240, loss 0.9752
step 250, loss 0.9873
step 260, loss 0.9331
step 270, loss 0.9510
step 280, loss 0.9441
step 290, loss 0.9502
step 300, loss 0.9736
step 310, loss 0.9803
step 320, loss 0.9240
step 330, loss 0.9548
step 340, loss 0.9451
step 350, loss 0.9193
step 360, loss 0.9970
step 370, loss 0.9637
step 380, loss 0.9898
step 390, loss 0.9647
step 400, loss 0.9828
step 410, loss 1.0382
step 420, loss 0.9878
step 430, loss 1.0303
step 440, loss 1.0345
step 450, loss 1.0110

In [13]:
def predict_bayesian(x, samples, model):
    preds = []
    offset = 0
    flat_vars = tf.concat([tf.reshape(v, [-1]) for v in model.trainable_variables], axis=0)
    shapes = [v.shape for v in model.trainable_variables]
    sizes = [tf.size(v) for v in model.trainable_variables]

    for s in samples:
        idx = 0
        for var, shape, size in zip(model.trainable_variables, shapes, sizes):
            new_val = tf.reshape(s[idx:idx+size], shape)
            var.assign(new_val)
            idx += size
        preds.append(model(x, training=False))
    return tf.reduce_mean(preds, axis=0), tf.math.reduce_std(preds, axis=0)

In [36]:
x_test = tf.random.normal([1, 128, 128, 3])
mean_pred, std_pred = predict_bayesian(x_test, samples[-50:], model)

In [38]:
print(mean_pred.shape, std_pred.shape)

(1, 8, 8, 5) (1, 8, 8, 5)
